# Increasing Performance

This chapter covers
- optimization for performance

At the example of the Gillespie simulation algorithm, we will discuss ways to
increase performance of an implementation.

We start with the usual imports:

In [3]:
# %pip install numpy scipy numba line_profiler

# multiprocessing & progress
import joblib
import tqdm

import numpy as np
import scipy.stats as st
import numba

# plotting
import matplotlib.pyplot as plt
figsize = (5,4)

# profiler
%load_ext line_profiler

And also redefined types and variables from the previous section.

In [5]:
from typing import TypedDict


class CRN(TypedDict):
    species: list[str]
    r_vec: np.ndarray
    state_update_vec: np.ndarray
    rate_constants: np.ndarray


species = ["mRNA", "protein"]

r_vec = np.array([
    [0, 0],  # DNA -> DNA + mRNA
    [1, 0],  # mRNA -> 0
    [1, 0],  # mRNA -> mRNA + protein
    [0, 1],  # protein -> 0
    ], dtype=int)

state_update_vec: np.ndarray = np.array(
    [
        [ 1,  0],  # DNA -> DNA + mRNA
        [-1,  0],  # mRNA -> 0
        [ 0,  1],  # mRNA -> mRNA + protein
        [ 0, -1],  # protein -> 0
    ],
    dtype=int,
)

rate_constants = np.array([0.5, 0.1, 0.2, 0.05], dtype=float)

example_crn: CRN = {
    "species": species,
    "r_vec": r_vec,
    "state_update_vec": state_update_vec,
    "rate_constants": rate_constants,
}

## Profiling

We use `%lprun` to profile line by line.
We use the `-f` flag to specify which function to profile linewise.


In [29]:
%lprun \
    -T lp_results.txt \
    -f gillespie_ssa \
    gillespie_ssa(example_crn, initial_state, time_points)



*** Profile printout saved to text file 'lp_results.txt'.


Timer unit: 1e-09 s

Total time: 0.019007 s
File: /var/folders/hd/g16tvgfd6pddg2qvrkrypykr0000gp/T/ipykernel_36476/1865262429.py
Function: gillespie_ssa at line 1

Line #      Hits         Time  Per Hit   % Time  Line Contents
     1                                           def gillespie_ssa(
     2                                                   crn: CRN,
     3                                                   initial_state: np.ndarray,
     4                                                   output_times: np.ndarray,
     5                                                   ) -> np.ndarray:
     6         1       1000.0   1000.0      0.0      update = crn["state_update_vec"]
     7                                           
     8                                               # allocate arrays
     9         1      98000.0  98000.0      0.5      output_states = np.empty((len(output_times), update.shape[1]), dtype=int)
    10         1       6000.0   6000.0      0.0      propensiti

About 80% of the time is spent doing draws.
Let us see how we can improve our draw speed by profiling the `gillespie_draw` function.

In [30]:
propensities_test = np.zeros(len(example_crn["rate_constants"]))
%lprun \
    -T lp_results.txt \
    -f gillespie_draw \
        [gillespie_draw(initial_state, propensities_test, example_crn) \
            for _ in range(10_000)]



*** Profile printout saved to text file 'lp_results.txt'.


Timer unit: 1e-09 s

Total time: 0.177377 s
File: /var/folders/hd/g16tvgfd6pddg2qvrkrypykr0000gp/T/ipykernel_36476/1844850471.py
Function: gillespie_draw at line 1

Line #      Hits         Time  Per Hit   % Time  Line Contents
     1                                           def gillespie_draw(
     2                                                   state: np.ndarray,
     3                                                   propensities: np.ndarray,
     4                                                   crn: CRN,
     5                                                   ) -> tuple[int, float]:
     6     10000  106620000.0  10662.0     60.1      update_propensity(propensities, state, crn["r_vec"], crn["rate_constants"])
     7     10000   15859000.0   1585.9      8.9      props_sum = propensities.sum()
     8     10000   20279000.0   2027.9     11.4      delta = np.random.exponential(1.0 / props_sum)
     9     10000   30875000.0   3087.5     17.4      rxn = sample_discrete(propensi

`update_propensity` is taking the most time. We will focus on speeding that up.


## Just-in-time compilation

A significant speed boost is achieved by [just-in-time compliation](https://en.wikipedia.org/wiki/Just-in-time_compilation) using [Numba](http://numba.pydata.org). To utilize this feature, you need to just-in-time compile (JIT) your propensity function. You can insist that everything is compiled (and therefore skip the comparably slow Python interpreter) by using the `@numba.njit` decorator. In many cases, that is all you have to do.

### Speed boost by JIT compilation with Numba

[Numba](https://numba.pydata.org) is a package that does LLVM optimized just-in-time compilation of Python code.  The speed-ups can be substantial.  We will use `numba` to compile the parts of the code that we can.  For many functions, we just need to decorate the function with

    @numba.jit()

If possible, we can use the `nopython=True` keyword argument to get more speed because the compiler can assume that all of the code is JIT-able.
Using

    @numba.njit
    
is equivalent to using `@numba.jit(nopython=True)`.

First, recall that we used the following `numpy` implementation.

```python
def update_propensity(
        propensities: np.ndarray,
        state: np.ndarray,
        r_vec: np.ndarray,
        rate_constants: np.ndarray) -> None:
    propensities[:] = rate_constants * (
        np.where(r_vec >= 1, state, 1) * np.where(r_vec == 2, state - 1, 1)
    ).prod(axis=1)
```

The implementation is already making good use of vectorization and the natively compiled `numpy` library, but the number of reactions is probably too small (in our case 4), for vectorization to pay off.
We will thus step back to a Python implementation and let the `numba` compilation do its job.
We also include the pure Python function for comparison with both.

In [ ]:
# pure Python
def update_propensity_python(
        propensities: np.ndarray,
        state: np.ndarray,
        r_vec: np.ndarray,
        rate_constants: np.ndarray) -> None:
    for i in range(len(propensities)):
        p = rate_constants[i]
        for s in range(len(state)):
            r = r_vec[i, s]
            if r >= 1:
                p *= state[s]
            if r >= 2:
                p *= state[s] - 1
        propensities[i] = p

# jit of pure Python
@numba.njit
def update_propensity_numba(
        propensities: np.ndarray,
        state: np.ndarray,
        r_vec: np.ndarray,
        rate_constants: np.ndarray) -> None:
    for i in range(len(propensities)):
        p = rate_constants[i]
        for s in range(len(state)):
            r = r_vec[i, s]
            if r >= 1:
                p *= state[s]
            if r >= 2:
                p *= state[s] - 1
        propensities[i] = p

In [33]:
propensities_test = np.zeros(len(example_crn["rate_constants"]))
state_test = np.array([10, 100], dtype=int)

print('numpy:')
%timeit \
    update_propensity(propensities_test, state_test, example_crn["r_vec"], example_crn["rate_constants"])

print('Python:')
%timeit \
    update_propensity_python(propensities_test, state_test, example_crn["r_vec"], example_crn["rate_constants"])

print('numba:')
%timeit \
    update_propensity_numba(propensities_test, state_test, example_crn["r_vec"], example_crn["rate_constants"])


numpy:
4.96 μs ± 156 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
Python:
1.9 μs ± 35.7 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)
numba:
341 ns ± 1.98 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)


We got a clear speed-up.
Now let us also speed up the sum and the discrete sampling.

In [34]:
@numba.njit
def sum_numba(ar):
    return ar.sum()

# Make dummy array for testing
ar = np.array([0.3, 0.4, 0.3, 0.2, 0.15])

print('numpy:')
%timeit ar.sum()

print('numba:')
%timeit sum_numba(ar)

numpy:
617 ns ± 4.9 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)
numba:
122 ns ± 2.22 ns per loop (mean ± std. dev. of 7 runs, 10,000,000 loops each)


We get another speed boost, though we should note that this speed boost in the sum is due to `numba` optimizing sums of a certain size.

Finally, we will speed up the sampling of the discrete distribution.  We will do this in two ways.  First, we notice that the division operation on the propensities took a fair amount of time when we profiled the code.  We do not need it; we can instead sample from an unnormalized discrete distribution.  Secondly, we can use `numba` to accelerate the while loop in the sampling.

In [35]:
@numba.njit
def sample_discrete_numba(probs, probs_sum):
    q: float = np.random.rand() * probs_sum

    i: int = 0
    p_sum: float = 0.0
    while p_sum < q:
        p_sum += probs[i]
        i += 1
    return i - 1


# Make dummy unnormalized probs
probs = np.array([0.1, 0.3, 0.4, 0.05, 0.15, 0.6])
probs_sum = sum_numba(probs)

print('Python:')
%timeit sample_discrete(probs)

print("numba:")
%timeit sample_discrete_numba(probs, probs_sum)

Python:
528 ns ± 1.18 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)
numba:
152 ns ± 1.08 ns per loop (mean ± std. dev. of 7 runs, 10,000,000 loops each)


We get a speed-up of about a factor of three.  Let's now make a new `gillespie_draw()` function that is faster.

In [37]:
def gillespie_draw_fast(
        state: np.ndarray,
        propensities: np.ndarray,
        r_vec: np.ndarray,
        rate_constants: np.ndarray) -> tuple[int, float]:
    
    # use fastest one:
    update_propensity_numba(propensities, state, r_vec, rate_constants)
    
    # use fastest one:
    props_sum = sum_numba(propensities)

    # reuse props_sum here and in next function call
    delta = np.random.exponential(1.0 / props_sum)

    # use fastest one:
    rxn = sample_discrete_numba(propensities, props_sum)
    
    return rxn, delta


propensities_test = np.zeros(len(example_crn["rate_constants"]))
state_test = np.array([10, 100], dtype=int)

print('gillespie_draw:')
%timeit gillespie_draw(state_test, propensities_test, example_crn)

print('gillespie_draw_fast:')
%timeit gillespie_draw_fast(state_test, propensities_test, example_crn["r_vec"], example_crn["rate_constants"])


gillespie_draw:
7.51 μs ± 179 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
gillespie_draw_fast:
1.05 μs ± 15.4 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)


Let's adjust our SSA function to include the fast draws.

In [38]:
def gillespie_ssa_fast(
        crn: CRN,
        initial_state: np.ndarray,
        output_times: np.ndarray,
        ) -> np.ndarray:
    update = crn["state_update_vec"]
    
    # allocate arrays
    output_states = np.empty((len(output_times), update.shape[1]), dtype=int)
    propensities = np.zeros(update.shape[0])

    r_vec = example_crn["r_vec"]
    rate_constants = example_crn["rate_constants"]

    i_time: int = 1
    i: int = 0

    # initial time, state, and output
    t = output_times[0]
    state = initial_state.copy()
    state_prev = state.copy()
    output_states[0, :] = state

    # event loop
    while i < len(output_times):
        while t < output_times[i_time]:
            # 2 random draws
            event, dt = gillespie_draw_fast(state, propensities, r_vec, rate_constants)

            # update
            state_prev = state.copy()
            state += update[event, :]
            t += dt

        # passed an output time
        i = np.sum(output_times <= t)
        output_states[i_time : min(i, len(output_times))] = state_prev
        i_time = i
    return output_states


Now we can test the speed of the two SSAs.

In [39]:
print('gillespie_ssa:')
%timeit gillespie_ssa(example_crn, initial_state, time_points)

print('gillespie_ssa_fast:')
%timeit gillespie_ssa_fast(example_crn, initial_state, time_points)


gillespie_ssa:
4.66 ms ± 44.8 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
gillespie_ssa_fast:
1.18 ms ± 4.6 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


We get a speed boost from JIT-compiled propensity, sum, and sampling. The Python loop in the SSA itself is still interpreted. To go further, we JIT-compile the entire simulation by passing the CRN arrays directly into a fully `@numba.njit` function.


In [23]:
@numba.njit
def gillespie_draw_numba(
        state: np.ndarray,
        propensities: np.ndarray,
        r_vec: np.ndarray,
        rate_constants: np.ndarray) -> tuple[int, float]:
    update_propensity_numba(propensities, state, r_vec, rate_constants)
    props_sum = propensities.sum()
    delta = np.random.exponential(1.0 / props_sum)
    rxn = sample_discrete_numba(propensities, props_sum)
    return rxn, delta


@numba.njit
def gillespie_ssa_numba(
        r_vec: np.ndarray,
        state_update_vec: np.ndarray,
        rate_constants: np.ndarray,
        population_0: np.ndarray,
        time_points: np.ndarray) -> np.ndarray:
    pop_out = np.empty((len(time_points), state_update_vec.shape[1]), dtype=np.int64)
    i_time = 1
    i = 0
    t = time_points[0]
    state = population_0.copy()
    pop_out[0, :] = state
    propensities = np.zeros(r_vec.shape[0])
    while i < len(time_points):
        while t < time_points[i_time]:
            event, dt = gillespie_draw_numba(
                state, propensities, r_vec, rate_constants
            )
            state_prev = state.copy()
            state += state_update_vec[event, :]
            t += dt
        i = np.searchsorted((time_points > t).astype(np.int64), 1)
        for j in range(i_time, min(i, len(time_points))):
            pop_out[j, :] = state_prev
        i_time = i
    return pop_out


Now let's test the speed of all three of our functions.

In [ ]:
print('gillespie_ssa:')
%timeit gillespie_ssa(example_crn, initial_state, time_points)

print('gillespie_ssa_fast:')
%timeit gillespie_ssa_fast(example_crn, initial_state, time_points)

print('gillespie_ssa_numba:')
%timeit gillespie_ssa_numba(example_crn["r_vec"], example_crn["state_update_vec"], example_crn["rate_constants"], initial_state, time_points)


gillespie_ssa:
4.76 ms ± 41.2 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)

gillespie_ssa_fast:
1.13 ms ± 12.1 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)

gillespie_ssa_numba:
66.9 μs ± 11 μs per loop (mean ± std. dev. of 7 runs, 1 loop each)


The speed up is significant, so we should probably use Numba'd code.

## Parallel Gillespie simulations

Sampling by the Gillespie algorithm is trivially parallelizable. We can use the `joblib` module to parallelize the computation. Syntactically, we need to specify a function that takes a single argument. Below, we set up a parallel calculation of Gillespie simulations for our specific example.

In [25]:
def gillespie_parallel(
        crn: CRN,
        population_0: np.ndarray,
        time_points: np.ndarray,
        n_simulations: int,
        n_jobs: int) -> np.ndarray:
    populations = joblib.Parallel(n_jobs=n_jobs)(
        joblib.delayed(gillespie_ssa_numba)(
            crn["r_vec"], crn["state_update_vec"],
            crn["rate_constants"], population_0, time_points
        ) for _ in range(n_simulations)
    )
    return np.array(populations)


We are paying some overhead in setting up the parallelization.  Let's time it to see how we do with parallelization.

In [26]:
n_simulations = 1000

print('numba Gillespie SSA:')
%timeit [gillespie_ssa_numba(example_crn["r_vec"], example_crn["state_update_vec"], example_crn["rate_constants"], initial_state, time_points) for _ in range(n_simulations)]

print('\nParallel numba Gillespie SSA:')
print(f'core count = {joblib.cpu_count()}')
%timeit gillespie_parallel(example_crn, initial_state, time_points, n_simulations, 4)


numba Gillespie SSA:
65.2 ms ± 1.28 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)

Parallel numba Gillespie SSA:
core count = 8
109 ms ± 3.62 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


We get another speed boost. This brings our total speed boost from the optimization to near two orders of magnitude.

### Heuristics to further improve speed

If we insist on *exact* sampling out of a probability distribution defined by a master question, we can get significant speed boosts by switching to the [Gibson and Bruck algorithm](http://doi.org/10.1021/jp993732q), especially for more complicated systems and propensity functions. If we are willing to *approximately* sample out of the probability distribution, there are many fast, approximate methods (e.g., [Salis and Kaznessis](https://doi.org/10.1063%2F1.1835951)) available.

## Approximations: Tau-leaping

We can further speed-up the computational time of the algorithms by approximating the solution. One such method is called [Tau-leaping](https://en.wikipedia.org/wiki/Tau-leaping).

<hr>
**License & Attribution**: This page is from material by [Michael Elowitz and Justin Bois](https://biocircuits.github.io/) (© 2021–2025), licensed under [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/), with minor modifications.